# TriangleGrid to VoronoiGridPlus workflow

This notebook demonstrates the more natural grid-building workflow for this project:

1. define a domain with `TriangleGrid`
2. add one or more refinement regions
3. inspect the prepared region points
4. build the Triangle mesh
5. create `VoronoiGridPlus` from the mesh
6. build and run a simple MF6 model from `SimpleModelConfig`


In [ ]:
from pathlib import Path

import myflopy as mf

In [ ]:
workspace = Path.cwd().resolve().parents[1] / "artifacts" / "triangle_voronoi_workflow"
workspace.mkdir(parents=True, exist_ok=True)

tri = mf.TriangleGrid(model_ws=str(workspace), angle=30)
tri.set_domain_rectangle(x_dist=2000, y_dist=1200, origin=(0, 0))
tri.add_region_circle(
    center_coords=(900, 600),
    radius=220,
    max_area=10_000,
    label="target_zone",
    priority=2,
)
tri.add_region_rectangle(
    origin=(1200, 250),
    x_dist=450,
    y_dist=300,
    max_area=20_000,
    label="secondary_zone",
    priority=1,
)


In [ ]:
tri.preview_regions()

In [ ]:
tri.get_region_points()

In [ ]:
tri.build()
vor = mf.VoronoiGridPlus(tri)
vor.gdf_vorPolys.head()

In [ ]:
boundary_cells = vor.get_grid_edge()

config = mf.SimpleModelConfig(
    vor=vor,
    name="tri_demo",
    mf_folder_path=workspace,
    nper=1,
    nlay=1,
    grid_type="disv",
    top=[50.0] * vor.ncpl,
    bottom=[[0.0] * vor.ncpl],
    initial_heads=[50.0] * vor.ncpl,
    k=[10.0] * vor.ncpl,
    save_specific_discharge=False,
    boundary_mode="chd",
    boundary_cells=boundary_cells,
    boundary_head=[50.0] * len(boundary_cells),
    sto_steady={0: True},
    sto_transient={},
)

model = mf.build_simple_model(config)
success, output = model.run_simulation()
success

In [ ]:
model.all_heads[["elev", "geometry"]].head()

## Notes

- `preview_regions()` shows the prepared refinement regions and the claimed area for each one
- `get_region_points()` shows the points Triangle will use to enforce region cell sizes
- keep `SimpleModelConfig.name` at 16 characters or fewer because MF6 enforces that limit
- once the grid exists, the rest of the workflow is the same as any other `VoronoiGridPlus`-based model
